In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from datetime import datetime

# --- CONFIGURACIÓN ---
TICKERS = ['BTC-USD', 'MSTR']
MODEL_PATH = 'trading_bot_v3.pth'
SCALER_PATH = 'scaler.pkl'
WINDOW_SIZE = 60
INPUT_DIM = 6
HIDDEN_DIM = 32
NUM_LAYERS = 1
THRESHOLD_BUY = 0.55
THRESHOLD_SELL = 0.45

# --- 1. DEFINICIÓN DEL MODELO ---
class TradingBotLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(TradingBotLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0)
        self.dropout = nn.Dropout(0.5) 
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return self.sigmoid(out)

# --- 2. OBTENCIÓN DE DATOS ROBUSTA ---
def get_live_data():
    print("--- INICIANDO DESCARGA DE DATOS ---")
    
    # Descarga por separado
    try:
        btc_raw = yf.download('BTC-USD', period="6mo", interval="1h", auto_adjust=True, progress=False)
        mstr_raw = yf.download('MSTR', period="6mo", interval="1h", auto_adjust=True, progress=False)
    except Exception as e:
        print(f"Error descarga: {e}")
        return None

    # Extracción de columna Close
    def extract_close(df):
        if isinstance(df.columns, pd.MultiIndex):
            try: return df.xs('Close', axis=1, level=0).iloc[:, 0]
            except: 
                try: return df.xs('Close', axis=1, level=1).iloc[:, 0]
                except: return df.iloc[:, 0]
        else:
            return df['Close'] if 'Close' in df.columns else df.iloc[:, 0]

    btc_series = extract_close(btc_raw)
    mstr_series = extract_close(mstr_raw)

    # Sincronización Temporal
    def normalize_index(series):
        if series.index.tz is None: series.index = series.index.tz_localize('UTC')
        else: series.index = series.index.tz_convert('UTC')
        series.index = series.index.floor('h')
        return series[~series.index.duplicated(keep='last')]

    btc_series = normalize_index(btc_series)
    mstr_series = normalize_index(mstr_series)

    # Fusión
    df = pd.concat([btc_series, mstr_series], axis=1, join='inner')
    df.columns = ['BTC', 'MSTR']
    df = df.dropna()

    if len(df) < 60:
        print("ERROR: Menos de 60 horas coincidentes.")
        return None

    # Feature Engineering
    df['lret_btc'] = np.log(df['BTC'] / df['BTC'].shift(1))
    df['lret_mstr'] = np.log(df['MSTR'] / df['MSTR'].shift(1))
    df['ratio'] = df['MSTR'] / df['BTC']
    df['ratio_mean'] = df['ratio'].rolling(window=60).mean()
    df['ratio_std'] = df['ratio'].rolling(window=60).std()
    df['z_score'] = (df['ratio'] - df['ratio_mean']) / df['ratio_std']
    
    # --- CORRECCIÓN AQUÍ ---
    # Guardamos las columnas de precio ('BTC', 'MSTR') también, para poder imprimirlas luego
    cols_to_keep = ['BTC', 'MSTR', 'lret_btc', 'lret_mstr', 'ratio', 'z_score', 'ratio_mean', 'ratio_std']
    
    df_final = df[cols_to_keep].dropna()
    
    print(f"Datos procesados listos: {len(df_final)} filas.")
    return df_final

# --- 3. EJECUCIÓN PRINCIPAL ---
def main():
    try:
        model = TradingBotLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS)
        model.load_state_dict(torch.load(MODEL_PATH))
        model.eval()
        scaler = joblib.load(SCALER_PATH)
        print("Cerebro cargado correctamente.")
    except:
        print("ERROR CRÍTICO: No se encuentran los archivos .pth o .pkl")
        return

    df = get_live_data()
    if df is None: return

    # --- CORRECCIÓN AQUÍ TAMBIÉN ---
    # El modelo SOLO quiere ver las 6 variables numéricas, no los precios.
    # Definimos explícitamente qué columnas son "features" para la IA.
    model_features = ['lret_btc', 'lret_mstr', 'ratio', 'z_score', 'ratio_mean', 'ratio_std']
    
    # Extraemos solo esas 6 columnas para el tensor
    last_sequence = df.iloc[-WINDOW_SIZE:][model_features].values
    
    # Escalamos
    last_sequence_scaled = scaler.transform(last_sequence)
    
    # Tensor
    input_tensor = torch.from_numpy(last_sequence_scaled).float().unsqueeze(0)
    
    with torch.no_grad():
        probability = model(input_tensor).item()
        
    print("\n" + "="*40)
    print(f"FECHA Y HORA: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    # Ahora sí funciona porque 'BTC' y 'MSTR' siguen en el dataframe 'df'
    print(f"PRECIO BTC:   ${df.iloc[-1]['BTC']:.2f}")
    print(f"PRECIO MSTR:  ${df.iloc[-1]['MSTR']:.2f}")
    print(f"Z-SCORE ACT:  {df.iloc[-1]['z_score']:.4f}")
    print("-" * 40)
    print(f"PROBABILIDAD DEL MODELO: {probability:.4f}")
    print("-" * 40)
    
    if probability > THRESHOLD_BUY:
        print(">>> RECOMENDACIÓN: COMPRAR (LONG) 🟢")
    elif probability < THRESHOLD_SELL:
        print(">>> RECOMENDACIÓN: VENDER (SHORT/CASH) 🔴")
    else:
        print(">>> RECOMENDACIÓN: NEUTRAL (ESPERAR) ⚪")
    print("="*40 + "\n")

if __name__ == "__main__":
    main()

Modelo y Escalador cargados correctamente.
--- INICIANDO DESCARGA DE DATOS (MODO ROBUSTO) ---
Descargando BTC...
Descargando MSTR...
Filas alineadas correctamente: 832
Datos procesados listos para IA: 773 muestras.

FECHA Y HORA: 2026-03-05 00:22


KeyError: 'BTC'